# FE747 Data Analytics in Finance
## Drawdown Model — Logistic Regression

**Instructions:**
1. Run Cell 1 — loads data and connects Drive (run once)
2. Run Cell 2 — opens the model dashboard
3. Enter your Strategy Name, set parameters, click **▶ Run Model**
4. When satisfied, click **💾 Export** to save to Drive

> Both cells are collapsed by default. Click the ▶ button at left to run each cell.

In [ ]:
# @title ⚙️ Setup & Data Load (run once — do not edit) { display-mode: "form" }
import pandas as pd, numpy as np, io, zipfile, requests, re
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
import ipywidgets as widgets
from IPython.display import display, clear_output
import json, os, datetime, calendar
from google.colab import drive

drive.mount('/content/drive')

DRIVE_PATH  = '/content/drive/My Drive/MSFin_Python/final_project/'
MODELS_PATH = os.path.join(DRIVE_PATH, 'models/')
os.makedirs(MODELS_PATH, exist_ok=True)

def fetch_ff_zip(url):
    hdrs = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'Referer': 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html'
    }
    resp = requests.get(url, headers=hdrs, timeout=30)
    resp.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(resp.content))
    csv_name = [n for n in z.namelist() if n.upper().endswith('.CSV')][0]
    raw = z.read(csv_name).decode('latin-1')
    lines = raw.splitlines()
    date_re = re.compile(r'^\s*\d{8}\s*,')
    first_data_idx = None
    for i, line in enumerate(lines):
        if date_re.match(line):
            first_data_idx = i
            header_row_idx = i - 1
            break
    col_header = lines[header_row_idx]
    data_lines = [col_header]
    for line in lines[first_data_idx:]:
        if date_re.match(line):
            data_lines.append(line)
        elif line.strip() == '':
            continue
        else:
            break
    df = pd.read_csv(io.StringIO('\n'.join(data_lines)), index_col=0)
    df.columns = [c.strip() for c in df.columns]
    df.index = pd.to_datetime(df.index.astype(str).str.strip(), format='%Y%m%d')
    df.index.name = 'Date'
    df = df.apply(pd.to_numeric, errors='coerce') / 100
    return df

def fetch_vix():
    url = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=VIXCLS'
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    df = pd.read_csv(io.StringIO(resp.text), index_col=0, parse_dates=True)
    df.columns = ['VIX']
    df['VIX'] = pd.to_numeric(df['VIX'], errors='coerce')
    return df['VIX'].dropna()

URL_FF3 = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_daily_CSV.zip'
URL_MOM = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Momentum_Factor_daily_CSV.zip'

print("Loading data...")
ff3 = fetch_ff_zip(URL_FF3)
mom = fetch_ff_zip(URL_MOM)
vix = fetch_vix()

ff = ff3.join(mom, how='inner').join(vix, how='inner')
ff = ff.loc['1990-01-01':].dropna()

MKT_RF_COL = [c for c in ff.columns if 'mkt' in c.lower() or 'rm' in c.lower()][0]
SMB_COL    = [c for c in ff.columns if 'smb'  in c.lower()][0]
HML_COL    = [c for c in ff.columns if 'hml'  in c.lower()][0]
RF_COL     = [c for c in ff.columns if 'rf'   in c.lower() and 'mkt' not in c.lower()][0]
MOM_COL    = [c for c in ff.columns if 'mom'  in c.lower() or 'wml' in c.lower()][0]

WINDOW = 63
ff['Mkt']     = ff[MKT_RF_COL] + ff[RF_COL]
ff['Mkt_idx'] = (1 + ff['Mkt']).cumprod()

RAW_COLS = {MKT_RF_COL:'Mkt_RF', SMB_COL:'SMB', HML_COL:'HML',
            MOM_COL:'Mom', RF_COL:'RF', 'VIX':'VIX'}
for raw_col, nice in RAW_COLS.items():
    ff[f'{nice}_raw'] = ff[raw_col]
    ff[f'{nice}_5d']  = ff[raw_col].rolling(5).mean()
    ff[f'{nice}_21d'] = ff[raw_col].rolling(21).mean()

FEAT_COLS = [c for c in ff.columns if c.endswith(('_raw','_5d','_21d'))]
for fc in FEAT_COLS:
    ff[fc] = ff[fc].shift(1)

ff_base = ff.dropna(subset=FEAT_COLS).copy()
print(f"\u2705 Ready: {len(ff_base):,} obs  ({ff_base.index[0].date()} \u2192 {ff_base.index[-1].date()})")
print("\u2705 Drive connected  |  models/ folder ready")
print("\nRun Cell 2 to open the model dashboard.")

In [ ]:
# @title \U0001f3af Drawdown Model Dashboard { display-mode: "form" }

C_TN = '#f4a7a7'; C_FP = '#c0392b'; C_FN = '#a8d5b0'; C_TP = '#27ae60'

# ── Widgets ───────────────────────────────────────────────────────────────────
strategy_name = widgets.Text(
    value='', placeholder='e.g. GFC_Focused',
    description='Strategy Name:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)
year_widget = widgets.Dropdown(
    options=list(range(ff_base.index[0].year, ff_base.index[-1].year - 2)),
    value=1995, description='Start Year:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='180px')
)
month_widget = widgets.Dropdown(
    options=[(datetime.date(2000, m, 1).strftime('%B'), m) for m in range(1, 13)],
    value=1, description='Start Month:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='180px')
)
duration_slider = widgets.IntSlider(
    value=48, min=36, max=60, step=1,
    description='Duration (months):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)
dd_input = widgets.BoundedFloatText(
    value=20.0, min=5.0, max=40.0, step=1.0,
    description='DD Level (%):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='180px')
)
threshold_input = widgets.BoundedFloatText(
    value=0.10, min=0.05, max=0.95, step=0.01,
    description='Threshold:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='180px')
)
run_button = widgets.Button(
    description='\u25b6 Run Model', button_style='primary',
    layout=widgets.Layout(width='140px', height='36px')
)
export_button = widgets.Button(
    description='\U0001f4be Export', button_style='success',
    layout=widgets.Layout(width='140px', height='36px')
)
status_label = widgets.HTML(value='<i>Set parameters and click Run Model.</i>')
output_area  = widgets.Output()

clf = None
ff_model = None

display(widgets.VBox([
    widgets.HTML("<b>FE747 Data Analytics in Finance | Drawdown Model</b>"),
    widgets.HBox([strategy_name]),
    widgets.HBox([year_widget, month_widget, duration_slider]),
    widgets.HBox([dd_input, threshold_input, run_button, export_button]),
    status_label,
    output_area
]))

# ── Helpers ───────────────────────────────────────────────────────────────────
def build_target(dd_level_pct):
    threshold = -(dd_level_pct / 100.0)
    mkt_arr   = ff_base['Mkt_idx'].values
    n         = len(mkt_arr)
    target    = np.full(n, np.nan)
    for i in range(n - WINDOW):
        seg         = mkt_arr[i : i + WINDOW]
        rolling_max = np.maximum.accumulate(seg)
        target[i]   = 1 if (seg / rolling_max - 1).min() <= threshold else 0
    return target

def draw_histogram(ax, y_true, y_prob, thresh, prob_label, title):
    bin_edges = np.linspace(0, 1, 21)
    for actual, col_l, col_r, lbl in [
        (0, C_TN, C_FP, 'No DD'),
        (1, C_FN, C_TP, 'DD')
    ]:
        probs_cls = y_prob[y_true == actual]
        ax.hist(probs_cls[probs_cls <  thresh], bins=bin_edges,
                color=col_l, alpha=0.8, edgecolor=col_r, lw=0.8)
        ax.hist(probs_cls[probs_cls >= thresh], bins=bin_edges,
                color=col_r, alpha=0.55, edgecolor=col_r, lw=0.8, label=lbl)
    ax.axvline(thresh, color='black', ls='--', lw=1.5, label=f'Thresh={thresh:.2f}')
    ax.set_title(title, fontweight='bold', fontsize=8)
    ax.set_xlabel(prob_label, fontsize=7)
    ax.set_ylabel('Count', fontsize=7)
    ax.legend(fontsize=6); ax.set_xlim(0, 1)
    ax.tick_params(labelsize=7)

def draw_confusion(ax, y_true, y_pred_bin, label0, label1, title):
    if len(y_true) == 0:
        ax.text(0.5, 0.5, 'No data\nfor this period',
                ha='center', va='center', fontsize=10, color='gray',
                transform=ax.transAxes)
        ax.set_title(title, fontweight='bold', fontsize=8)
        ax.axis('off'); return

    cm_mat         = confusion_matrix(y_true, y_pred_bin, labels=[0,1])
    tn, fp, fn, tp = cm_mat.ravel()
    n_obs          = len(y_true)
    acc            = (tp + tn) / n_obs
    prec0 = tn / (tn + fn) if (tn + fn) > 0 else 0
    rec0  = tn / (tn + fp) if (tn + fp) > 0 else 0
    prec1 = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec1  = tp / (tp + fn) if (tp + fn) > 0 else 0

    cell_colours = [[C_TN, C_FP],[C_FN, C_TP]]
    cell_values  = [[tn, fp],[fn, tp]]
    cell_tags    = [['TN','FP'],['FN','TP']]

    ax.set_xlim(0, 2); ax.set_ylim(0, 2.6); ax.invert_yaxis()
    for r in range(2):
        for c in range(2):
            rect = mpatches.FancyBboxPatch(
                (c+0.04, r+0.04), 0.92, 0.82,
                boxstyle='round,pad=0.02',
                facecolor=cell_colours[r][c], edgecolor='white', lw=2)
            ax.add_patch(rect)
            count = cell_values[r][c]
            tag   = cell_tags[r][c]
            ax.text(c+0.50, r+0.22, f'{tag}  {count}',
                    ha='center', va='center',
                    fontsize=11, fontweight='bold', color='white')
            ax.text(c+0.50, r+0.50, f'{count/n_obs*100:.1f}% obs',
                    ha='center', va='center', fontsize=8, color='white')
            ax.text(c+0.50, r+0.74, 'Pred 0' if c==0 else 'Pred 1',
                    ha='center', va='center', fontsize=7, color='white', style='italic')

    stats = (f'Acc: {acc:.1%}  Base: {y_true.mean():.1%}  N={n_obs:,}\n'
             f'Prec  0:{prec0:.1%}  1:{prec1:.1%}\n'
             f'Recall 0:{rec0:.1%}  1:{rec1:.1%}')
    ax.text(1.0, 2.30, stats, ha='center', va='center',
            fontsize=7.5, family='monospace',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#d6d6d6',
                      edgecolor='#888', lw=1))
    ax.set_xticks([]); ax.set_yticks([0.5, 1.5])
    ax.set_yticklabels([f'Act 0 ({label0})', f'Act 1 ({label1})'], fontsize=7)
    ax.set_title(title, fontweight='bold', fontsize=8, pad=4)
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.tick_params(length=0)

# ── Run Model ─────────────────────────────────────────────────────────────────
def run_model(btn):
    global clf, ff_model, X_train, y_train, X_oos, y_oos

    yr       = year_widget.value
    mo       = month_widget.value
    dur      = duration_slider.value
    dd_level = dd_input.value
    thresh   = threshold_input.value
    sname    = strategy_name.value.strip() or 'UNNAMED'

    train_start = pd.Timestamp(yr, mo, 1)
    end_mo  = mo + dur - 1
    end_yr  = yr + (end_mo - 1) // 12
    end_mo  = (end_mo - 1) % 12 + 1
    last_day = calendar.monthrange(end_yr, end_mo)[1]
    train_end = pd.Timestamp(end_yr, end_mo, last_day)
    oos_start = train_end + pd.offsets.BDay(1)



    with output_area:
        clear_output(wait=True)
        print(f"Computing {dd_level:.0f}% drawdown labels...")
        target = build_target(dd_level)
        ff_model = ff_base.copy()
        ff_model['DrawdownEvent'] = target
        ff_model = ff_model.dropna(subset=['DrawdownEvent'])
        ff_model['is_insample'] = (
            (ff_model.index >= train_start) & (ff_model.index <= train_end)
        ).astype(int)

        status_label.value = (
            f'<b>Strategy:</b> {sname} &nbsp;|&nbsp; '
            f'<b>Train:</b> {train_start.date()} \u2192 {train_end.date()} ({dur}mo) &nbsp;|&nbsp; '
            f'<b>Pre-IS:</b> {ff_model.index[0].date()} \u2192 {(train_start - pd.offsets.BDay(1)).date()} &nbsp;|&nbsp; '
            f'<b>Post-IS:</b> {oos_start.date()} \u2192 {ff_model.index[-1].date()} &nbsp;|&nbsp; '
            f'<b>DD:</b> {dd_level:.0f}% &nbsp;|&nbsp; <b>Thresh:</b> {thresh:.2f}'
        )

        pre_mask   = ff_model.index < train_start
        train_mask = (ff_model.index >= train_start) & (ff_model.index <= train_end)
        post_mask  = ff_model.index >= oos_start

        X_train = ff_model.loc[train_mask, FEAT_COLS].astype(float)
        y_train = ff_model.loc[train_mask, 'DrawdownEvent'].astype(float)
        X_pre   = ff_model.loc[pre_mask,   FEAT_COLS].astype(float)
        y_pre   = ff_model.loc[pre_mask,   'DrawdownEvent'].astype(float)
        X_post  = ff_model.loc[post_mask,  FEAT_COLS].astype(float)
        y_post  = ff_model.loc[post_mask,  'DrawdownEvent'].astype(float)

        if len(X_train) < 200 or int(y_train.sum()) < 5:
            print("\u274c Window too short or insufficient events. Adjust parameters.")
            return

        clf = LogisticRegression(
            penalty='l1', solver='liblinear', C=20.0,
            class_weight='balanced', max_iter=1000, random_state=42
        )
        clf.fit(X_train.values, y_train)

        # Probabilities for all periods
        def get_probs(X):
            if len(X) == 0: return np.array([]), np.array([])
            p = clf.predict_proba(X.values)[:, 1]
            return p, (p >= thresh).astype(int)

        p_is,  pred_is  = get_probs(X_train)
        p_pre, pred_pre = get_probs(X_pre)
        p_post,pred_post= get_probs(X_post)

        log_odds = X_train.values @ clf.coef_[0] + clf.intercept_[0]
        label0   = 'No DD'
        label1   = f'DD\u2265{dd_level:.0f}%'
        prob_lbl = f'P(DD\u2265{dd_level:.0f}%)'

        # ── FIGURE 1: Top row — Coefficients | IS Hist | Pre Hist | Post Hist
        fig1, axes1 = plt.subplots(1, 4, figsize=(16, 3))
        fig1.suptitle(
            f'FE747 | Drawdown Model: {sname} | DD\u2265{dd_level:.0f}% | '
            f'Train: {train_start.date()} \u2192 {train_end.date()} | Thresh={thresh:.2f}',
            fontsize=9, fontweight='bold'
        )

        # Col 1: Coefficients
        ax = axes1[0]
        coef_s  = pd.Series(clf.coef_[0], index=FEAT_COLS)
        coef_df = (pd.DataFrame({'Feature': coef_s.index, 'Coeff': coef_s.values})
                     .assign(Abs=lambda d: d['Coeff'].abs())
                     .nlargest(12, 'Abs').sort_values('Coeff'))
        colours_b = [C_TP if c > 0 else C_FP for c in coef_df['Coeff']]
        ax.barh(coef_df['Feature'], coef_df['Coeff'],
                color=colours_b, edgecolor='white', lw=0.5)
        ax.axvline(0, color='black', lw=0.8)
        ax.set_title('Coefficients (Log-Odds)', fontweight='bold', fontsize=8)
        ax.set_xlabel('Log-odds', fontsize=7)
        ax.tick_params(labelsize=6)

        # Col 2: IS Histogram
        draw_histogram(axes1[1], y_train.values, p_is, thresh, prob_lbl,
                       f'In-Sample Histogram\n{train_start.date()} \u2192 {train_end.date()}')

        # Col 3: Pre-IS Histogram
        if len(y_pre) > 0:
            draw_histogram(axes1[2], y_pre.values, p_pre, thresh, prob_lbl,
                           f'Pre-IS Histogram\n{ff_model.index[0].date()} \u2192 {(train_start-pd.offsets.BDay(1)).date()}')
        else:
            axes1[2].text(0.5, 0.5, 'No pre-IS\ndata', ha='center', va='center',
                          fontsize=10, color='gray', transform=axes1[2].transAxes)
            axes1[2].set_title('Pre-IS Histogram', fontweight='bold', fontsize=8)
            axes1[2].axis('off')

        # Col 4: Post-IS Histogram
        if len(y_post) > 0:
            draw_histogram(axes1[3], y_post.values, p_post, thresh, prob_lbl,
                           f'Post-IS Histogram\n{oos_start.date()} \u2192 {ff_model.index[-1].date()}')
        else:
            axes1[3].text(0.5, 0.5, 'No post-IS\ndata', ha='center', va='center',
                          fontsize=10, color='gray', transform=axes1[3].transAxes)
            axes1[3].set_title('Post-IS Histogram', fontweight='bold', fontsize=8)
            axes1[3].axis('off')

        plt.tight_layout()
        plt.show()

        # ── FIGURE 2: Bottom row — Sigmoid | IS CM | Pre CM | Post CM ────
        fig2, axes2 = plt.subplots(1, 4, figsize=(16, 3))

        # Col 1: Sigmoid
        ax = axes2[0]
        z_range  = np.linspace(log_odds.min(), log_odds.max(), 300)
        p_range  = 1 / (1 + np.exp(-z_range))
        thresh_z = np.log(thresh / (1 - thresh)) if 0 < thresh < 1 else 0

        z_min, z_max = log_odds.min(), log_odds.max()
        z_range_width = z_max - z_min
        x_offset = z_range_width * 0.12

        ax.plot(z_range, p_range, color='steelblue', lw=2, label=prob_lbl)
        ax.axhline(thresh, color='red', ls='--', lw=1.2)
        ax.axvline(thresh_z, color='red', ls=':', lw=1, alpha=0.6)
        # Mark threshold point
        ax.plot(thresh_z, thresh, 'ro', ms=5)
        ax.annotate(f'z={thresh_z:.2f}\np={thresh:.2f}',
            xy=(thresh_z, thresh), xytext=(thresh_z - x_offset, thresh + 0.15),
            fontsize=7, color='red',
            arrowprops=dict(arrowstyle='->', color='red', lw=0.8))
        # Mark 0.50 point
        z50 = 0.0
        ax.plot(z50, 0.50, 'bs', ms=5)

        y_offset = -0.12  # below right

        ax.annotate(f'z=0.00\np=0.50',
            xy=(z50, 0.50), xytext=(z50 + x_offset, 0.50 + y_offset),
            fontsize=7, color='steelblue',
            arrowprops=dict(arrowstyle='->', color='steelblue', lw=0.8))

        ax.set_title('Sigmoid Curve', fontweight='bold', fontsize=8)
        ax.set_xlabel('Linear Score z', fontsize=7)
        ax.set_ylabel(prob_lbl, fontsize=7)
        ax.set_ylim(-0.03, 1.05)
        ax.tick_params(labelsize=7)

        # Col 2: IS Confusion
        draw_confusion(axes2[1], y_train.values, pred_is, label0, label1,
                       f'In-Sample Confusion\n{train_start.date()} \u2192 {train_end.date()}')

        # Col 3: Pre-IS Confusion
        draw_confusion(axes2[2],
                       y_pre.values if len(y_pre) > 0 else np.array([]),
                       pred_pre if len(pred_pre) > 0 else np.array([]),
                       label0, label1,
                       f'Pre-IS Confusion\n{ff_model.index[0].date()} \u2192 {(train_start-pd.offsets.BDay(1)).date()}')

        # Col 4: Post-IS Confusion
        draw_confusion(axes2[3],
                       y_post.values if len(y_post) > 0 else np.array([]),
                       pred_post if len(pred_post) > 0 else np.array([]),
                       label0, label1,
                       f'Post-IS Confusion\n{oos_start.date()} \u2192 {ff_model.index[-1].date()}')

        plt.tight_layout()
        plt.show()
        print(f"\u2705 Model run complete — click \U0001f4be Export to save to Drive")

run_button.on_click(run_model)

# ── Export ────────────────────────────────────────────────────────────────────
def export_model(btn):
    with output_area:
        global clf, ff_model
        sname = strategy_name.value.strip() or 'UNNAMED'
        if clf is None:
            print("\u274c Run the model first before exporting.")
            return

        def build_coef_block(clf, feat_cols):
            coef_map = dict(zip(feat_cols, clf.coef_[0]))
            return {"intercept": round(float(clf.intercept_[0]), 6),
                    **{k: round(float(v), 6) for k, v in coef_map.items()}}

        payload = {
            "team_id":   sname,
            "submission_timestamp": pd.Timestamp.now().isoformat(timespec='seconds'),
            "fit_window": {
                "start_year":      int(year_widget.value),
                "start_month":     int(month_widget.value),
                "duration_months": int(duration_slider.value)
            },
            "drawdown_model": {
                "threshold_pct":      float(dd_input.value),
                "probability_cutoff": float(threshold_input.value),
                "features":           FEAT_COLS,
                "coefficients":       build_coef_block(clf, FEAT_COLS)
            }
        }
        json_path = os.path.join(MODELS_PATH, f'{sname}_drawdown_model.json')
        with open(json_path, 'w') as f:
            json.dump(payload, f, indent=2)
        print(f"\u2705 JSON saved: {json_path}")

        X_all    = ff_model[FEAT_COLS].astype(float)
        prob_all = clf.predict_proba(X_all.values)[:, 1]
        prob_df  = pd.DataFrame({
            'P_Drawdown':  prob_all,
            'Signal':      (prob_all >= threshold_input.value).astype(int),
            'is_insample': ff_model['is_insample'].values
        }, index=ff_model.index)
        prob_df.index.name = 'Date'
        csv_path = os.path.join(MODELS_PATH, f'{sname}_drawdown_probs.csv')
        prob_df.to_csv(csv_path)
        print(f"\u2705 Probs saved: {csv_path}")
        print(f"\u2705 Export complete \u2014 check your models/ folder in Drive")

export_button.on_click(export_model)